In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import mysql.connector
from getpass import getpass

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
from pathlib import Path

csv_path = Path(
    r"C:\Users\aatif\OneDrive\Desktop\healthcare-readmission-analysis\data\processed\feature_engineered_diabetes_data.csv"
)

print("File exists:", csv_path.exists())

File exists: True


In [6]:
from getpass import getpass
import mysql.connector

mysql_password = getpass("Enter your MySQL root password: ")

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password=mysql_password,
    database="healthcare_readmission"
)

print("Connected to MySQL successfully!")

Enter your MySQL root password:  ········


Connected to MySQL successfully!


In [9]:
# Load the feature-engineered dataset
df = pd.read_csv(csv_path, low_memory=False)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

# Check the MySQL table
cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM diabetes_readmission")
mysql_columns = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT COUNT(*) FROM diabetes_readmission")
existing_rows = cursor.fetchone()[0]

print("MySQL table columns:", len(mysql_columns))
print("Existing rows:", existing_rows)

# Compare column names and order
if list(df.columns) == mysql_columns:
    print("SUCCESS: CSV and MySQL columns match exactly.")
else:
    print("WARNING: CSV and MySQL columns do not match.")
    print("CSV-only columns:", sorted(set(df.columns) - set(mysql_columns)))
    print("MySQL-only columns:", sorted(set(mysql_columns) - set(df.columns)))

cursor.close()

Dataset shape: (101766, 57)
Number of columns: 57
MySQL table columns: 57
Existing rows: 0
SUCCESS: CSV and MySQL columns match exactly.


True

In [21]:


# Prepare missing values for MySQL
df = df.astype(object).where(pd.notna(df), None)

# Build the INSERT statement using the CSV column names
columns = ", ".join(f"`{col}`" for col in df.columns)
placeholders = ", ".join(["%s"] * len(df.columns))

insert_sql = f"""
    INSERT INTO diabetes_readmission ({columns})
    VALUES ({placeholders})
"""

# Convert pandas values into Python-compatible values
def prepare_value(value):
    if value is None:
        return None
    if hasattr(value, "item"):
        value = value.item()
    if pd.isna(value):
        return None
    return value

batch_size = 500
cursor = conn.cursor()

try:
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]

        rows = [
            tuple(prepare_value(v) for v in row)
            for row in batch_df.itertuples(index=False, name=None)
        ]

        cursor.executemany(insert_sql, rows)
        conn.commit()

        if (start // batch_size) % 10 == 0:
            print(f"Processed {min(start + batch_size, len(df)):,} / {len(df):,} rows")

    print("Import completed!")

except Exception as e:
    conn.rollback()
    print("Import stopped:", e)
    raise

finally:
    cursor.close()

Processed 500 / 101,766 rows
Processed 5,500 / 101,766 rows
Processed 10,500 / 101,766 rows
Processed 15,500 / 101,766 rows
Processed 20,500 / 101,766 rows
Processed 25,500 / 101,766 rows
Processed 30,500 / 101,766 rows
Processed 35,500 / 101,766 rows
Processed 40,500 / 101,766 rows
Processed 45,500 / 101,766 rows
Processed 50,500 / 101,766 rows
Processed 55,500 / 101,766 rows
Processed 60,500 / 101,766 rows
Processed 65,500 / 101,766 rows
Processed 70,500 / 101,766 rows
Processed 75,500 / 101,766 rows
Processed 80,500 / 101,766 rows
Processed 85,500 / 101,766 rows
Processed 90,500 / 101,766 rows
Processed 95,500 / 101,766 rows
Processed 100,500 / 101,766 rows
Import completed!


In [22]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM diabetes_readmission")
print("Imported rows:", cursor.fetchone()[0])

cursor.execute("""
    SELECT readmitted_30, COUNT(*)
    FROM diabetes_readmission
    GROUP BY readmitted_30
""")

print("\n30-day readmission target:")
for row in cursor.fetchall():
    print(row)

cursor.close()

Imported rows: 101766

30-day readmission target:
(0, 90409)
(1, 11357)


True

In [23]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT encounter_id) AS unique_encounters
    FROM diabetes_readmission
""")

print(cursor.fetchone())

cursor.close()

(101766, 101766)


True